# Practice 59 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Retail quarterly performance

`stores` below records `revenue` ($k) and `footfall` (daily visitors) for six stores across three quarters.

1. Reshape into long format — one row per `(store_id, region, quarter)`. No syntax reminder.
2. Rank all stores by their mean revenue across the three quarters, lowest to highest. Use `np.argsort` on the grouped means.
3. Use `np.corrcoef` to check whether footfall and revenue move together across all store–quarter pairs. Does more foot traffic drive more revenue here?

In [14]:
stores = pd.DataFrame({
    'store_id':    [1, 2, 3, 4, 5, 6],
    'region':      ['North', 'North', 'South', 'South', 'West', 'West'],
    'revenue_Q1':  [120, 95, 140, 105, 88, 112],
    'revenue_Q2':  [135, 102, 128, 118, 94, 119],
    'revenue_Q3':  [142, 110, 155, 130, 99, 125],
    'footfall_Q1': [1800, 1500, 2100, 1700, 1400, 1750],
    'footfall_Q2': [2000, 1650, 1950, 1900, 1500, 1850],
    'footfall_Q3': [2100, 1750, 2200, 2000, 1600, 1900],
})


# Your code here
store_long = pd.wide_to_long(
    stores, 
    stubnames=['revenue','footfall'],
    i = ['store_id','region'],
    j = 'quarter',
    suffix=r'\w+',
    sep = '_'
)

sm = store_long.groupby(level='store_id')['revenue'].mean()
print(sm.iloc[np.argsort(sm)])

print(np.corrcoef(store_long['revenue'], store_long['footfall'])[0,1])
print('highly correlated positively')


store_id
5     93.666667
2    102.333333
4    117.666667
6    118.666667
1    132.333333
3    141.000000
Name: revenue, dtype: float64
0.9861941280442101
highly correlated positively


---

## Level 2 — Taxis

Use `sns.load_dataset('taxis')`.

**New: `margins=True`** — adds an `'All'` row and column to a crosstab, showing marginal totals:

```python
pd.crosstab(df['a'], df['b'], margins=True)
# 'All' column = total per row; 'All' row = total per column; bottom-right = grand total
```

It also works with `normalize` — `'All'` then shows marginal proportions (the `'All'` column under `normalize='index'` will always be 1.0).

1. Build a crosstab of `color` vs `payment`, with `margins=True`. Which taxi color runs more rides overall? For cash payments, which color is more common?
2. Build a crosstab of `pickup_borough` vs `color`, normalized by row, with `margins=True`. In which borough is yellow the smallest share of rides?
3. Add `tip_pct = tip / fare`. Drop rows where `fare` is 0 or `tip` is null. For each `payment` type, use `np.percentile` to find the 25th, 50th, and 75th percentile of `tip_pct`. Which payment type has the widest spread?

In [40]:
taxis = sns.load_dataset('taxis')

# Your code here
p = pd.crosstab(taxis['color'], taxis['payment'], margins=True)
print(p)
print(p.loc[p.index != 'All','All'].idxmax(), ' runs more rides overall')
print (p.loc[p.index != 'All', 'cash'].idxmax(), " is more common for cash payments")

pp = pd.crosstab(taxis['pickup_borough'], taxis['color'], normalize='index', margins=True)
print(pp)
print(f"in {pp['yellow'].idxmin()} is yellow the smallest share of rides")

taxis['tip_pct'] = taxis['tip']/taxis['fare']
taxis = taxis[taxis['fare']!=0].dropna(subset = ['tip'])

tg = taxis.groupby('payment')['tip_pct'].apply(lambda x: np.percentile(x, [25,75]))
print(tg)
print(tg.apply(lambda x: x[1]- x[0]).idxmax(),'has the widest spred')

payment  cash  credit card   All
color                           
green     400          577   977
yellow   1412         4000  5412
All      1812         4577  6389
yellow  runs more rides overall
yellow  is more common for cash payments
color              green    yellow
pickup_borough                    
Bronx           0.838384  0.161616
Brooklyn        0.817232  0.182768
Manhattan       0.055809  0.944191
Queens          0.438356  0.561644
All             0.152646  0.847354
in Bronx is yellow the smallest share of rides
payment
cash                              [0.0, 0.0]
credit card    [0.18181818181818182, 0.3065]
Name: tip_pct, dtype: object
credit card has the widest spred


---

## Level 3 — Car crashes

`sns.load_dataset('car_crashes')` has per-state crash statistics. Key columns: `total`, `speeding`, `alcohol`, `not_distracted`, `ins_premium`, `ins_losses`, `abbrev`.

No sub-steps — you decide the approach.

1. Which predictor — `speeding`, `alcohol`, or `not_distracted` — correlates most strongly with `total` crashes? Report all three `np.corrcoef` values.

2. Use `np.percentile` to compute the 33rd and 67th percentile of `total` as bin edges. Create a `crash_tier` column (`'low'`, `'mid'`, `'high'`) with `pd.cut` using `[-np.inf, p33, p67, np.inf]` as the bins. What is the mean `ins_premium` per tier — does riskier driving cost more to insure?

3. Use `np.argsort` to find the 5 states (`abbrev`) with the highest `ins_losses`. Are these the same 5 states with the most `total` crashes?

In [73]:
crashes = sns.load_dataset('car_crashes')

# Your code here

ic = ['speeding','alcohol','not_distracted']
ps = pd.Series([np.corrcoef(crashes[x],crashes['total'])[0,1] for x in ic], index=ic)
print(ps)
print(ps.idxmax(),'correlates most strongly with total crashes')

be = np.percentile(crashes['total'], q = [33,67])
crashes['crash_tier'] = pd.cut(crashes['total'], bins = [-np.inf] + list(be) + [np.inf], labels=['low','mid','high'])
cp = crashes.groupby('crash_tier', observed=True)['ins_premium'].mean()
print(cp)
print('not necessarily cost more for riskier driving')

tpl = crashes.iloc[np.argsort(-crashes['ins_losses'])]['abbrev'][:5]
tpt = crashes.iloc[np.argsort(-crashes['total'])]['abbrev'][:5]
print(tpl)
print(tpt)
set(tpl) == set(tpt)
print('not the same')


speeding          0.611548
alcohol           0.852613
not_distracted    0.827560
dtype: float64
alcohol correlates most strongly with total crashes
crash_tier
low     948.080588
mid     810.740000
high    902.052353
Name: ins_premium, dtype: float64
not necessarily cost more for riskier driving
18    LA
20    MD
36    OK
6     CT
4     CA
Name: abbrev, dtype: object
40    SC
34    ND
48    WV
3     AR
26    MT
Name: abbrev, dtype: object
not the same
